# CLOPS_v on Qiskit Serverless

This notebook runs the CLOPS benchmark defined in [Wack et al., *Scale, Quality, and Speed: three key attributes to measure the performance of near-term quantum computers*, arXiv:2110.14108](https://arxiv.org/abs/2110.14108), referred to here as **CLOPS_v**. The benchmark client runs on [Qiskit Serverless](https://quantum.cloud.ibm.com/docs/en/guides/serverless), inside IBM Quantum's cloud, so this machine can disconnect while it runs.

1. Generate **M = 100** parameterized Quantum Volume templates of width and depth **D = log2(QV)**, and compile them once before timing starts.
2. Draw each template's first parameters from a pseudo-random generator.
3. Run **K = 10** rounds. In every round after the first, each template's parameters are seeded from that template's previous measured output, so a template's round k cannot start until its round k−1 result is back.
4. Run every circuit with **S = 100** shots, at the system default repetition delay.

```
CLOPS_v = M × K × S × D / time_taken
```

`time_taken` is measured on the server clock: from the reported start of the first job (queue time excluded) to the reported finish of the last job.

**Requirements:**

- an IBM Quantum Premium, Flex or On-Prem plan;
- permission to upload custom Qiskit Functions (`function-custom.write`);
- `qiskit-ibm-catalog` and `qiskit-serverless` on this machine.

In [ ]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [ ]:
from qiskit_ibm_catalog import QiskitFunction, QiskitServerless

# uses the saved account; pass name=, token= or instance= to choose another
serverless = QiskitServerless()

## Upload the function

The function directory `function/` holds the benchmark, `clops_v.py`, and the serverless entrypoint, `run_clops_v.py`. Re-run this cell after changing either file.

In [ ]:
function = serverless.upload(
    QiskitFunction(
        title="clops-v",
        entrypoint="run_clops_v.py",
        working_dir="./function/",
        description="CLOPS_v benchmark (arXiv:2110.14108)",
    )
)
function

## Configure and run

The defaults are the operating point used for the reported results:

- **Backend and quantum volume:** `ibm_phoenix` at QV512, so D = 9.
- **Qubits:** the 3×3 grid used for its quantum volume measurement.
- **Rounds and shots:** the paper's M=100, K=10, S=100. Only this point is labelled `CLOPS_v(QV<n>)`.
- **Pipelines:** `pipelines=5`. The templates are split into that many interleaved job chains, which the protocol allows as long as each template's own rounds stay in order.

**On another backend,** set `backend_name` and `quantum_volume`. Then either pass that device's own set of D connected qubits, or use `qubits=None` to select the backend's reported layer fidelity chain of length D. The order of `qubits` is used as the initial layout.

**Repeat runs:** the reported results summarize several runs (best, mean and range). Set `runs` to repeat; the submitted runs use the backend one Session at a time.

In [ ]:
arguments = dict(
    backend_name="ibm_phoenix",
    quantum_volume=512,
    templates=100,  # M
    parameter_updates=10,  # K
    shots=100,  # S
    pipelines=5,  # interleaved job chains
    qubits=[0, 1, 2, 10, 11, 12, 20, 21, 22],  # 3x3 grid on ibm_phoenix
    seed=None,  # set for reproducible templates
)
runs = 1

jobs = [function.run(**arguments) for _ in range(runs)]
for job in jobs:
    print("serverless job id:", job.job_id)

You can close this notebook now and come back later. To reconnect to submitted jobs:

```python
jobs = [serverless.job(job_id) for job_id in ["<serverless job id>", ...]]
```

In [ ]:
[job.status() for job in jobs]

## Results

This waits for each job to finish, then saves every successful result to `results/`.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
results = []
for job in jobs:
    result = job.result(cadence=10)
    if "error" in result:
        print(job.job_id, "failed:", result["error"])
        continue
    results.append(result)
    start = datetime.fromisoformat(result["t_start"]).strftime("%Y-%m-%d_%H_%M_%S")
    fname = results_dir / (
        "clops_v_%s_%s_P%d.json" % (result["backend_name"], start, result["pipelines"])
    )
    with open(fname, "w") as fout:
        json.dump(result, fout, indent=2)
    print(
        "%s = %d  (%.1f s)  saved %s"
        % (result["label"], result["clops_v"], result["total_time_s"], fname)
    )

In [ ]:
result = results[-1]
print("%s = %d" % (result["label"], result["clops_v"]))
print(
    "time taken: %.1f s  (%s -> %s)"
    % (result["total_time_s"], result["t_start"], result["t_end"])
)
print("qubits:", result["qubits"], "(%s)" % result["qubit_set_source"])
print("interface:", result["execution_interface"])
print("environment:", result["environment"])

### Client side detail

These figures help diagnose a slow run:

- `client_submit_s`: total time spent submitting jobs.
- `client_wait_s`: total time spent waiting for results.
- `client_prepare_s`: time spent building each round's parameters and programs.
- `qpu_chunk_time_s`: the sum of the execution windows the Executor reports for each job.

Only `clops_v` above is the benchmark result.

In [ ]:
bd = result["breakdown"]
keys = (
    "qpu_chunk_time_s",
    "client_prepare_s",
    "client_submit_s",
    "client_wait_s",
    "client_total_s",
)
{k: bd[k] for k in keys}

## Summarize repeated runs

This summarizes the saved results for each operating point: the best value, the mean and standard deviation, the range, and N. It also prints a row ready to paste into the README's results table.

In [ ]:
%run summarize_results.py results

If a run failed, its logs usually say why:

In [ ]:
print(jobs[-1].logs()[-5000:])